# Segmentation

This notebook will explore the possibility of training an algorithm to segment a document into sentence-like units. The effort was originally prompted with a view to segmenting Ælfric, and corpus choices will reflect that aim.

In [48]:
import os,glob,re,json
from lxml import etree
from nltk.tokenize import PunktSentenceTokenizer, word_tokenize

In [ ]:
parser = etree.XMLParser(remove_blank_text=True, resolve_entities=True)

doec_path = '../DOEC/xml-corpus/'
echoe_path = '../echoe/xml/'

# Normalization should take both DOEC and ECHOE orthography into account.
# We will ignore questions around ampersand (i.e. that it should be "ond"
# or "et" in some contexts).
def normalize(target):
    matrix = {
        '&': 'and',
        'ð': 'þ',
        'ę': 'æ',
        'á': 'a',
        'ǽ': 'æ',
        'é': 'e',
        'í': 'i',
        'ó': 'o',
        'ú': 'u',
        'ý': 'y',
        'ẏ': 'y',
        'ƿ': 'w',
        'ſ': 's',
        '': 's', # descending s
        'v': 'u',
        'j': 'i',
        '⁊': 'and',
        '·': '',
        '.': '',
        ':': '',
        ';': '',
        ',': '',
        '!': '',
        '?': '',
        '-': '',
        '–': '',
        '—': '',
        '"': '',
        "'": '',
        '\n': ''

        }
    target = target.lower()
    for k,v in matrix.items():
        target = target.replace(k, v)
    return target

# We will discard unwanted elements from our ECHOE documents:
def simplify(branch):
    discard = ['abbr', 'am', 'del', 'sic', 'note', 'surplus']
    query = ['{http://www.tei-c.org/ns/1.0}' + i for i in discard]
    for element in query:
        for hit in branch.iter(element):
            hit.getparent().remove(hit)

# To load selected DOEC documents, we will load from a dictionary
# linking DOEC document identifiers with DOE short short titles.
# Note that many editions used for DOEC include rhetoric in their
# segmentation thinking, thus deprioritizing syntactic unity.
def load_from_toc(source_dict, doec_path):
    corpus = dict()
    for ref,title in source_dict.items():
        segments = []
        file = doec_path + ref + '.xml'
        tree = etree.parse(file, parser=parser)
        root = tree.getroot()
        text = root.find('.//text')
        for segment in text.iter('s'):
            # We will treat editorial (semi)colons as sentence boundaries:
            plaintext = re.split(':|;', etree.tostring(segment, method='text', encoding='unicode'))
            for sentence in plaintext:
                tokens = word_tokenize(normalize(sentence))
                segments.append(tokens)
        corpus[title] = segments
    return corpus

# At this stage we will load ECHOE in its entirety. ECHOE segmentation was performed
# manually, and not consistently on the whole, but an effort was made to make it
# syntactic rather than rhetorical. All but the shortest inquit-phrases are often
# their own unit (depending on source segmentation; see ECHOE documentation).
def load_echoe(echoe_path):
    corpus = dict()
    for doc in sorted(glob.glob(echoe_path + '*.xml')):
        identifier = os.path.basename(doc).replace('.xml', '')
        segments = []
        tree = etree.parse(doc, parser=parser)
        root = tree.getroot()
        text = root.find('.//{http://www.tei-c.org/ns/1.0}text')
        simplify(text)
        for segment in text.iter('{http://www.tei-c.org/ns/1.0}s'):
            tokens = []
            for token in segment.iter('{http://www.tei-c.org/ns/1.0}w'):
            # If a word element is marked as the last part of a word, add its text content to the preceding token:
                if token.get('part') == 'F':
                    position = len(tokens)-1
                    tokens[position] = tokens[position] + normalize(etree.tostring(token, method='text', encoding='unicode'))
                else:
                    tokens.append(normalize(etree.tostring(token, method='text', encoding='unicode')))
            segments.append(tokens)
        corpus[identifier] = segments
    return(corpus)


In [32]:
with open('../aelfric/doec-aelfric-toc.json', 'r') as json_file:
    aelfric_toc = json.load(json_file)
    



In [37]:
aelfric = load_from_toc(aelfric_toc, doec_path)
echoe = load_echoe(echoe_path)
corpus = aelfric | echoe

Like all widely used sentence tokenizers, [Punkt Sentence Tokenizer](https://www.nltk.org/api/nltk.tokenize.punkt.html) relies primarily on punctuation and capitalization. So much so, in fact, that it takes unsegmented text as its training input. But if we turn our corpus into one continuous string with neither punctuation nor capitalization, there is no way it can learn how to segment it:

In [46]:
all_sentences = [inner for outer in echoe.values() for inner in outer]
all_tokens = [inner for outer in all_sentences for inner in outer]
corpus_string = ' '.join(all_tokens)

In [49]:
sentence_tokenizer = PunktSentenceTokenizer(corpus_string)

In [51]:

wulfstan = 'leofan men gecnawað þæt soð is ðeos woruld is on ofste and hit nealæcð þam ende and ðy hit is on worulde a swa leng swa wyrse and swa hit sceal nyde for folces synnan fram dæge to dæge ær antecristes tocyme yfelian swyðe and huru hit wyrð þænne egeslic and grimlic wide on worulde leofan men understandað eac georne þæt deofol þas þeode nu fela geara dwelode to swyðe and þæt lytle getrywða wæron mid mannum þeah hi wel spæcan and unrihta to fela ricsode on lande and næs a fela manna þe smeade ymbe þa bote swa georne swa man scolde ac dæghwamlice man ihte yfel æfter oðrum and unriht rærde and unlaga manege ealles to wide gynd ealle þas ðeode'
wulfstan_sentences = sentence_tokenizer.tokenize(wulfstan)
len(wulfstan_sentences)

1

Thus we are left to our own devices, having to train a classifier to determine the likelihood of any particular n-gram being immediately preceded or followed by a sentence boundary.

In [ ]:
ngram_size = 4
# (PICK UP HERE, add bayesian classifier)